In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# =========================
# 1) Install dependencies
# =========================
!pip install -q transformers accelerate sentencepiece pandas tqdm


In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm


In [ ]:
# Dataset & columns
DATASET_DIR = "/kaggle/input/towards-ds-20records/towards_data_science_100_articles.csv"   # <- your uploaded dataset
TITLE_COL   = "Title"
TEXT_COL    = "Content"
model_name = "sshleifer/distilbart-cnn-12-6"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to("cuda" if torch.cuda.is_available() else "cpu")


# Output
OUTPUT_CSV = "/kaggle/working/summarized.csv"

# Model
MODEL_ID = "sshleifer/distilbart-cnn-12-6"

# Chunking
MAX_INPUT_TOKENS = 1024   # model hard limit
CHUNK_OVERLAP     = 50    # token overlap between chunks

# Generation settings for each chunk
CHUNK_GEN_KWARGS = dict(
    max_length=142,
    min_length=56,
    num_beams=4,
    do_sample=False,
    no_repeat_ngram_size=3,
    length_penalty=2.0,
    early_stopping=True
)

# Generation settings for the final (merged) summary
FINAL_GEN_KWARGS = dict(
    max_length=200,
    min_length=80,
    num_beams=4,
    do_sample=False,
    no_repeat_ngram_size=3,
    length_penalty=2.0,
    early_stopping=True
)


In [ ]:
df = pd.read_csv("/kaggle/input/towards-ds-20records/towards_data_science_100_articles.csv")
df = df.dropna(subset=['Content'])  # Drop rows with null content


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID).to(device)
if device.type == "cuda":
    model.half()  # optional: speed & memory win on T4
model.eval()


In [ ]:
def chunk_text(text, tokenizer, max_tokens=1024, stride=20):
    tokens = tokenizer.encode(text, truncation=False)
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk = tokens[start:end]
        chunks.append(tokenizer.decode(chunk, skip_special_tokens=True))
        start += max_tokens - stride
    return chunks

def summarize_chunks(text, tokenizer, model):
    device = model.device
    chunks = chunk_text(text, tokenizer)
    summaries = []
    for chunk in chunks:
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True, max_length=1024).to(device)
        summary_ids = model.generate(**inputs, max_length=150, min_length=30, length_penalty=2.0, num_beams=4, early_stopping=True)
        summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        summaries.append(summary)
    return " ".join(summaries)


In [ ]:
tqdm.pandas()
df["summary"] = df["Content"].progress_apply(lambda x: summarize_chunks(x, tokenizer, model))


In [ ]:
df.to_csv("summarized_towards_ds.csv", index=False)
